## Setup
- `session`: works like a browser. It sends our User-Agent every time and keeps cookies.
- `get()`: every request goes through it. It waits at most 15 seconds, then pauses 1 second.
- `can_fetch()`: asks the site's robots.txt if a robot may visit a page.

> 💼 Write the rules (timeout, pause) **once** in `get()`, and every request follows them.

In [1]:
import json
import time
from pathlib import Path
from urllib import robotparser

import requests

USER_AGENT = "ESILV-Lab/1.0 (educational use; contact: student@example.com)"

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

output = Path("output")
output.mkdir(exist_ok=True)


def get(url, **options):
    """GET with our session and a timeout, then wait 1 second (be polite)."""
    response = session.get(url, timeout=15, **options)
    time.sleep(1)
    return response


def can_fetch(site, path):
    """Read the site's robots.txt and check one path."""
    robots = robotparser.RobotFileParser()
    robots.parse(get(site + "/robots.txt").text.splitlines())
    return robots.can_fetch(USER_AGENT, site + path)

## 1. Wikipedia: the User-Agent matters
Wikipedia only answers programs that say who they are.

- 🌐 https://en.wikipedia.org/wiki/Albert_Einstein

> 💼 It worked yesterday, no data today? Check `status_code`: 403 or 429 often means "no name, no data".

First **without** our User-Agent:

In [2]:
url = "https://en.wikipedia.org/wiki/Albert_Einstein"

r = requests.get(url, timeout=15)
print(r.status_code)
print(r.text)

403
Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



**403 Forbidden**: Wikipedia refuses the default name `python-requests`. Now with our User-Agent:

In [3]:
page = get(url)
print(page.status_code, page.headers["Content-Type"])
print(len(page.text), "characters")

200 text/html; charset=UTF-8
1834383 characters


The page title (the text in the browser tab) is between `<title>` and `</title>`.

> 💼 Check the title first: is it the page you wanted, or a login / *"Are you a robot?"* page?

In [4]:
start = page.text.find("<title>") + len("<title>")
end = page.text.find("</title>")
page.text[start:end]

'Albert Einstein - Wikipedia'

### Headers: what the server tells us
The server sends headers back too: the type of the page, how it was sent, when it last changed.

> 💼 `Last-Modified` did not change? Then the page did not change: no need to download it again.

In [5]:
for name in ["Content-Type", "Content-Encoding", "Content-Length", "Last-Modified"]:
    print(f"{name:17}", page.headers.get(name))

Content-Type      text/html; charset=UTF-8
Content-Encoding  gzip
Content-Length    298549
Last-Modified     Sun, 22 Nov 2026 23:06:58 GMT


### Robots: may we search?
**robots.txt** is a file where a site lists the pages robots must **not** visit (`Disallow:` lines):
https://en.wikipedia.org/robots.txt

- 🌐 A Wikipedia search: https://en.wikipedia.org/w/index.php?search=Einstein+relativity&fulltext=1
  The search words are in the part after `?` (the `params`).

> 💼 Before any scraping job, read the site's robots.txt and terms of use.

Before we send the search, we ask robots.txt:

In [ ]:
site = "https://en.wikipedia.org"
print("article page:", can_fetch(site, "/wiki/Albert_Einstein"))
print("search page :", can_fetch(site, "/w/index.php?search=Einstein+relativity&fulltext=1"))


article page: True
search page : False


Articles (`/wiki/`) are allowed. Search (`/w/`) is not, so we do **not** send the search.

### The same article in three languages
Only the start of the address changes:
[en](https://en.wikipedia.org/wiki/Albert_Einstein) · [fr](https://fr.wikipedia.org/wiki/Albert_Einstein) · [de](https://de.wikipedia.org/wiki/Albert_Einstein).
So one loop downloads all three.

> 💼 One product on `amazon.fr`, `amazon.de`, `amazon.it`: one loop is enough.

In [7]:
for lang in ["en", "fr", "de"]:
    r = get(f"https://{lang}.wikipedia.org/wiki/Albert_Einstein")
    title = r.text[r.text.find("<title>") + 7 : r.text.find("</title>")]
    print(lang, r.status_code, f"{len(r.text):>10,} characters |", title)

en 200  1,834,383 characters | Albert Einstein - Wikipedia
fr 200  1,011,156 characters | Albert Einstein — Wikipédia
de 200  1,025,225 characters | Albert Einstein – Wikipedia


The `session` kept the **cookies** these three sites gave us:

In [8]:
sorted({cookie.domain for cookie in session.cookies})

['.wikipedia.org', 'de.wikipedia.org', 'en.wikipedia.org', 'fr.wikipedia.org']

One cookie list **per site**: each cookie goes back **only** to the site that gave it.

**Plan B:** not on the English site? Take it from the French one.
ESILV has a French article but **no** English one:
[English (does not exist)](https://en.wikipedia.org/wiki/%C3%89cole_sup%C3%A9rieure_d%27ing%C3%A9nieurs_L%C3%A9onard-de-Vinci) · [French](https://fr.wikipedia.org/wiki/%C3%89cole_sup%C3%A9rieure_d%27ing%C3%A9nieurs_L%C3%A9onard-de-Vinci)

In [9]:
def first_found(title, languages):
    """Try each Wikipedia language in order. Return (language, response) of the first page that exists."""
    for lang in languages:
        r = get(f"https://{lang}.wikipedia.org/wiki/{title}")
        print("  try", lang, "->", r.status_code)
        if r.status_code == 200:
            return lang, r
    return None, None


for title in ["Albert_Einstein", "École_supérieure_d'ingénieurs_Léonard-de-Vinci"]:
    print(title)
    lang, r = first_found(title, ["en", "fr"])
    print("  found in:", lang)

Albert_Einstein
  try en -> 200
  found in: en
École_supérieure_d'ingénieurs_Léonard-de-Vinci
  try en -> 404
  try fr -> 200
  found in: fr


Tip: also save **where** each value came from (`source_lang = "fr"`).

A page that does not exist: https://en.wikipedia.org/wiki/Albert_Einstein_ESILV_does_not_exist

> 💼 Dead link? Write it in an error list and go on.

In [10]:
r = get("https://en.wikipedia.org/wiki/Albert_Einstein_ESILV_does_not_exist")
print(r.status_code, r.ok)

404 False


## 2. Project Gutenberg: download a book
Free e-books. The book page is for people; the plain text file is the whole book, for programs.

- 🌐 Einstein's *Relativity*: https://www.gutenberg.org/ebooks/30155
- 🤖 Plain text (*Other formats* → *Plain Text UTF-8*): https://www.gutenberg.org/ebooks/30155.txt.utf-8

> 💼 Downloading 1,000 books to study? Save each one to a file first.

May a robot use the search page? https://www.gutenberg.org/ebooks/search/?query=einstein

In [11]:
site = "https://www.gutenberg.org"
print("text file  :", can_fetch(site, "/ebooks/30155.txt.utf-8"))
print("search page:", can_fetch(site, "/ebooks/search/?query=einstein"))

text file  : True
search page: True


**No.** So we took the book number **30155** from the book page address, by hand.

> 💼 No search allowed? Get the numbers elsewhere: Open Library (section 3) gives Einstein's Gutenberg number `1630`,
> and robots may read https://www.gutenberg.org/ebooks/author/1630

In [12]:
book = get("https://www.gutenberg.org/ebooks/30155.txt.utf-8")
print(book.status_code, book.headers["Content-Type"])
print("asked :", "https://www.gutenberg.org/ebooks/30155.txt.utf-8")
print("got   :", book.url)

503 text/html; charset=utf-8
asked : https://www.gutenberg.org/ebooks/30155.txt.utf-8
got   : https://www.gutenberg.org/ebooks/30155.txt.utf-8


We asked for one address and arrived at another: a **redirect**. requests followed it; `r.history` shows the steps.

> 💼 Save `r.url` (where you really arrived), so two old links to the same page are not saved twice.

In [13]:
for step in book.history:
    print(step.status_code, step.url, "->", step.headers["Location"])

In [14]:
print(len(book.text.split()), "words")
print(book.text[:300])

94 words
<!DOCTYPE html> <html lang="en"> <head> <meta charset="utf-8"> <meta name="viewport" content="width=device-width, initial-scale=1"> <title>Temporarily Down for Maintenance</title> <style> body { margin: 0; font: 1.1em/1.6 Helvetica, Arial, sans-serif; color: #222; background: #f4f4f4; } main { max-w


In [15]:
(output / "relativity.txt").write_text(book.text, encoding="utf-8")
print("saved", output / "relativity.txt")

saved output/relativity.txt


## 3. Open Library: a JSON API
Add `.json` to an Open Library address and you get the same information as JSON, for programs.

- 🌐 https://openlibrary.org/authors/OL3175986A
- 🤖 https://openlibrary.org/authors/OL3175986A.json

> 💼 Look for a JSON version before reading HTML: it is simpler, and it doesn't break when the design changes.

In [20]:
author = None

In [21]:
if author == None:
    author = get("https://openlibrary.org/authors/OL3175986A.json").json()
    print(author["name"], "|", author["birth_date"], "-", author["death_date"])

Albert Einstein | 14. März 1879 - 18. April 1955


JSON can be **nested**: a dictionary inside a dictionary. Use two names to go inside: `author["remote_ids"]["wikidata"]`.

> 💼 `remote_ids` gives Einstein's number on other sites (Wikidata, Gutenberg): more data about the same person.

In [22]:
author["remote_ids"]

{'viaf': '75121530',
 'wikidata': 'Q937',
 'isni': '000000012281955X',
 'bookbrainz': '43021f4f-4fda-4b32-bbf0-f2601181e32d',
 'musicbrainz': 'c98c325e-7277-46e8-8b44-e3517f3e041a',
 'goodreads': '9810',
 'imdb': 'nm0251868',
 'lc_naf': 'n79022889',
 'librarything': 'einsteinalbert',
 'librivox': '1035',
 'project_gutenberg': '1630',
 'opac_sbn': 'CFIV035853',
 'amazon': 'B000AP7JOU'}

In [80]:
print("Wikidata id :", author["remote_ids"]["wikidata"])
print("Gutenberg id:", author["remote_ids"]["project_gutenberg"])

Wikidata id : Q937
Gutenberg id: 1630


### His books, with `params`
`params={"limit": 5}` asks only for the first 5. requests adds `?limit=5` to the address:
https://openlibrary.org/authors/OL3175986A/works.json?limit=5

> 💼 Sites send big lists in small groups (`limit=50`): your loop asks for group 1, then 2, then 3...

In [81]:
r = get("https://openlibrary.org/authors/OL3175986A/works.json", params={"limit": 5})
print(r.url)

works = r.json()
print(works["size"], "works in total. The first 5:")
for work in works["entries"]:
    print("-", repr(work["title"]))

https://openlibrary.org/authors/OL3175986A/works.json?limit=5
594 works in total. The first 5:
- 'Textos fundamentais da física moderna'
- 'Como vejo o mundo'
- '\x98A \x9cevolução da física'
- '\x98O \x9csignificado da relatividade'
- '\x98A \x9cbomba H princípio ou fim?'


Real data is **messy**: some titles hide invisible characters (`\x98`, `\x9c`). We remove them:

> 💼 Always clean data before you save it. Project 1 does this in exercise 9 (`product_row`).

In [82]:
for work in works["entries"]:
    title = work["title"].replace("\x98", "").replace("\x9c", "")
    print("-", title)

- Textos fundamentais da física moderna
- Como vejo o mundo
- A evolução da física
- O significado da relatividade
- A bomba H princípio ou fim?


### A photo (bytes)
An image is **bytes**, not text: we save it with `write_bytes`.
https://covers.openlibrary.org/a/id/5724799-M.jpg (`5724799` comes from `author["photos"]`)

> 💼 An AI that recognises book covers needs thousands of pictures: put this code in a loop.

In [83]:
photo_id = author["photos"][0]
photo = get(f"https://covers.openlibrary.org/a/id/{photo_id}-M.jpg")
print(photo.status_code, photo.headers["Content-Type"], len(photo.content), "bytes")

(output / "einstein.jpg").write_bytes(photo.content)
print("saved", output / "einstein.jpg")

200 image/jpeg 7468 bytes
saved output\einstein.jpg


A work that does not exist: https://openlibrary.org/works/OL0000000000W.json

In [84]:
r = get("https://openlibrary.org/works/OL0000000000W.json")
print(r.status_code, r.text)

404 {"error": "notfound", "key": "/works/OL0000000000W"}


## 4. Open Food Facts: ask only for what you need
A food database. Each product is found by its **barcode**. The API gives the same product as JSON.

- 🌐 Nutella: https://world.openfoodfacts.org/product/3017624010701
- 🤖 https://world.openfoodfacts.org/api/v3/product/3017624010701.json

> 💼 A shop app scans a barcode and shows the Nutri-Score: it asks only for what it shows.

In [85]:
site = "https://world.openfoodfacts.org"
print("API page :", can_fetch(site, "/api/v3/product/3017624010701.json"))
print("HTML page:", can_fetch(site, "/product/3017624010701"))

API page : False
HTML page: True


robots.txt says **no** to `/api/`, but the Open Food Facts documentation **asks** apps to use the API.
robots.txt is for robots that crawl the whole site; an API has its own rules, in its documentation.

The full answer is big. `params={"fields": ...}` asks only for the 4 fields we need:

In [86]:
url = "https://world.openfoodfacts.org/api/v3/product/3017624010701.json"

full = get(url)
small = get(url, params={"fields": "product_name,brands,quantity,nutriscore_grade"})

print("all fields:", len(full.content), "bytes")
print("4 fields  :", len(small.content), "bytes")
small.json()

all fields: 41281 bytes
4 fields  : 250 bytes


{'code': '3017624010701',
 'errors': [],
 'product': {'brands': 'Ferrero',
  'nutriscore_grade': 'e',
  'product_name': 'Nutella',
  'quantity': '400.0 g'},
 'result': {'id': 'product_found',
  'lc_name': 'Product found',
  'name': 'Product found'},
 'status': 'success',
 'warnings': []}

Less data to download, for you **and** for the server. Now three products:
[Nutella](https://world.openfoodfacts.org/product/3017624010701) ·
[Coca-Cola](https://world.openfoodfacts.org/product/5449000000996) ·
[Cristaline](https://world.openfoodfacts.org/product/3274080005003)

> 💼 An Excel file of barcodes: this loop adds the name and the Nutri-Score to each one.

In [87]:
barcodes = {"Nutella": "3017624010701", "Coca-Cola": "5449000000996", "Cristaline water": "3274080005003"}

for name, code in barcodes.items():
    r = get(f"https://world.openfoodfacts.org/api/v3/product/{code}.json",
            params={"fields": "product_name,nutriscore_grade"})
    product = r.json()["product"]
    print(f"{name:17} Nutri-Score: {product['nutriscore_grade'].upper()}")

Nutella           Nutri-Score: E
Coca-Cola         Nutri-Score: E
Cristaline water  Nutri-Score: A


A barcode that does not exist: https://world.openfoodfacts.org/product/3017624010702

> 💼 Put unknown barcodes in a "to check" list instead of leaving empty boxes.

In [88]:
r = get("https://world.openfoodfacts.org/api/v3/product/3017624010702.json", params={"fields": "product_name"})
print(r.status_code)
r.json()

404


{'code': '3017624010702',
 'errors': [{'field': {'id': 'code', 'value': '3017624010702'},
   'impact': {'id': 'failure', 'lc_name': 'Failure', 'name': 'Failure'},
   'message': {'id': 'product_not_found', 'lc_name': '', 'name': ''}}],
 'result': {'id': 'product_not_found',
  'lc_name': 'Product not found',
  'name': 'Product not found'},
 'status': 'failure',
 'warnings': []}

**404**, and the JSON body says why. Always read the status code **and** the body.

In [89]:
(output / "nutella.json").write_text(json.dumps(small.json(), indent=2), encoding="utf-8")
print("saved", output / "nutella.json")

saved output\nutella.json


## 5. Yahoo Finance: a site that protects itself
Some sites block programs. Let's see how Yahoo reacts.

- 🌐 https://finance.yahoo.com/quote/AAPL/

> 💼 Banks don't scrape Yahoo: they pay for an API. We do it here only to learn.

In [90]:
print("quote page:", can_fetch("https://finance.yahoo.com", "/quote/AAPL/"))

quote page: True


robots.txt allows it. First **without** our User-Agent:

In [91]:
r = requests.get("https://finance.yahoo.com/quote/AAPL/", timeout=15)
print(r.status_code, r.text[:60])

429 Edge: Too Many Requests


**429 Too Many Requests**, on the very first request. Now with our User-Agent:

In [92]:
quote = get("https://finance.yahoo.com/quote/AAPL/")
print(quote.status_code, len(quote.text), "characters")

if quote.ok:
    i = quote.text.find('data-testid="qsp-price"')
    print(quote.text[i : i + 80])
else:
    print("Blocked:", quote.status_code)

200 1166852 characters
data-testid="qsp-price">348.51 </span> <span class="priceChange txt-negative yf-


It may work (200), or Yahoo may block you too.
If a site blocks you, **stop**: no fake browser, no faster retries. Use an official API.